# 04_SES_Feature_Engineering_V2
Data-Driven SES using LinkedIn + StackOverflow + XGBoost + SHAP

In [6]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
import shap


C:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Datasets

In [7]:
job_skills = pd.read_csv('job_skills.csv')
linkedin = pd.read_csv('linkedin_job_postings.csv')
stack = pd.read_csv('survey_results_public.csv', low_memory=False)


## Demand Score from LinkedIn

In [8]:
all_skills=[]

for row in job_skills['job_skills'].dropna():
    all_skills.extend([x.strip() for x in str(row).split(',')])

demand_df = pd.Series(all_skills).value_counts().reset_index()
demand_df.columns=['skill','demand_score']
demand_df.head()


,skill,demand_score
0,Communication,368293
1,Teamwork,226266
2,Leadership,184341
3,Customer service,166209
4,Communication skills,116260


## Developer Interest from StackOverflow

In [9]:
def extract_counts(series):
    counter = Counter()
    for row in series.dropna():
        for item in str(row).split(';'):
            counter[item.strip()] += 1
    return counter

interest_counter = Counter()

cols=[
'LanguageWantToWorkWith',
'DatabaseWantToWorkWith',
'PlatformWantToWorkWith',
'WebframeWantToWorkWith',
'ToolsTechWantToWorkWith',
'MiscTechWantToWorkWith'
]

for col in cols:
    interest_counter.update(extract_counts(stack[col]))

interest_df = pd.DataFrame(
    interest_counter.items(),
    columns=['skill','developer_interest']
)

interest_df.head()


,skill,developer_interest
0,Bash/Shell (all shells),13744
1,Go,13837
2,HTML/CSS,20721
3,Java,10668
4,JavaScript,23774


## Build Master Skill Dataset

In [10]:
master = demand_df.merge(
    interest_df,
    on='skill',
    how='outer'
).fillna(0)

master.head()


,skill,demand_score,developer_interest
0,,14.0,0.0
1,"""20 Tools of Process Control""",1.0,0.0
2,"""2nd Approval"" Opportunities",1.0,0.0
3,"""6 days on and 2 days off guaranteed"" schedule",1.0,0.0
4,"""A school"" or ""C school""",1.0,0.0


## Normalize Features

In [11]:
scaler = MinMaxScaler()

for col in ['demand_score','developer_interest']:
    master[col] = scaler.fit_transform(master[[col]])

master.head()


,skill,demand_score,developer_interest
0,,0.000038,0.0
1,"""20 Tools of Process Control""",0.000003,0.0
2,"""2nd Approval"" Opportunities",0.000003,0.0
3,"""6 days on and 2 days off guaranteed"" schedule",0.000003,0.0
4,"""A school"" or ""C school""",0.000003,0.0


## Create Proxy Target
Replace later with actual yearly skill growth when available.

In [12]:
master['future_growth_target'] = (
    0.6 * master['demand_score'] +
    0.4 * master['developer_interest']
)


## Train XGBoost

In [13]:
X = master[['demand_score','developer_interest']]
y = master['future_growth_target']

model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

model.fit(X,y)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

## Feature Importance

In [14]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})

importance.sort_values(
    'importance',
    ascending=False
)


,feature,importance
1,developer_interest,0.685897
0,demand_score,0.314103


## SHAP Analysis

In [15]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

shap_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': np.abs(shap_values).mean(axis=0)
})

shap_importance['weight'] = (
    shap_importance['importance'] /
    shap_importance['importance'].sum()
)

shap_importance.sort_values(
    'weight',
    ascending=False
)


,feature,importance,weight
0,demand_score,0.000020,0.710534
1,developer_interest,0.000008,0.289466


## Data-Driven SES

In [16]:
weights = dict(
    zip(
        shap_importance['feature'],
        shap_importance['weight']
    )
)

master['SES'] = (
    weights.get('demand_score',0) * master['demand_score'] +
    weights.get('developer_interest',0) * master['developer_interest']
) * 10

master[['skill','SES']].head()


,skill,SES
0,,0.000270
1,"""20 Tools of Process Control""",0.000019
2,"""2nd Approval"" Opportunities",0.000019
3,"""6 days on and 2 days off guaranteed"" schedule",0.000019
4,"""A school"" or ""C school""",0.000019


## Classification

In [17]:
def classify(score):
    if score < 3:
        return 'High Risk'
    elif score < 6:
        return 'Stable'
    elif score < 8:
        return 'Growing'
    return 'Future-Proof'

master['classification'] = master['SES'].apply(classify)


## Top SES Skills

In [18]:
master.sort_values(
    'SES',
    ascending=False
)[['skill','SES','classification']].head(50)


,skill,SES,classification
746815,Communication,7.105345,Growing
2971512,Teamwork,4.365269,Stable
1764684,Leadership,3.556425,Stable
2490369,Python,3.244961,Stable
904391,Customer service,3.206611,Stable
1032730,Docker,3.015292,Stable
2663353,SQL,2.893840,High Risk
1681466,JavaScript,2.781960,High Risk
2341361,PostgreSQL,2.676154,High Risk
1460024,HTML/CSS,2.299773,High Risk


## Export Files

In [19]:
master.to_csv('master_skill_features.csv',index=False)

master.sort_values(
    'SES',
    ascending=False
).to_csv('ses_rankings.csv',index=False)

shap_importance.to_csv(
    'ses_feature_weights.csv',
    index=False
)

print('Exports completed')


Exports completed
